# 과제 10 · 09 · 규칙으로 경로를 나누고 초안은 횟수 제한 안에서 다시 쓰기

## 현업 시나리오
모든 이벤트에 초안을 쓰면 안 됩니다. 예측과 판정이 어긋나면 사람에게 넘기고(escalation), 판정 기준에 해당하지 않는 경고는 점검만 권합니다(inspect_only). 초안을 쓰는 경로에서는 검사에 실패한 초안을 다시 쓰게 하되, 끝없이 다시 쓰게 둘 수는 없습니다.

## 학습 목표
- SOP 의 escalation 조건과 처리 경로를 규칙 함수로 만든다.
- `add_conditional_edges` 로 경로마다 다른 노드로 보낸다.
- 검사 실패 시 초안 노드로 되돌아가되, 횟수 상한에서 멈춘다.

## 직접 조립
완성된 App을 가져오지 않습니다. 아래에서 작은 fixture와 핵심 함수를 직접 만듭니다.

### 1단계 · 경로를 규칙으로

In [ ]:
# SOP-EA-01 6장(escalation 조건)과 9장(처리 경로)을 그대로 함수로 옮깁니다. LLM 에게 묻지 않습니다.
# 카드는 판정 결과까지 붙은 모양입니다. met 은 판정 기준이 성립한 유형입니다.
def practice_route(card):
    reasons = []
    if len(card["candidates"]) >= 2:
        reasons.append("ESC-2")
    if card["met"] and card["predicted"] not in card["met"]:
        reasons.append("ESC-3")
    if card["evidence"] == 0:
        reasons.append("ESC-4")
    if not card["met"] and card["severity"] == "alarm":
        reasons.append("ESC-7")
    if reasons:
        return "escalation", reasons
    if not card["met"]:
        return "inspect_only", ["SOP-EA-01 8"]
    return "grounded_draft", []

practice_cards = {
    "S01 HDF 정탐": {"predicted": "HDF", "candidates": ["HDF"], "met": ["HDF"], "severity": "warning", "evidence": 9},
    "S08 예측≠판정": {"predicted": "OSF", "candidates": ["OSF"], "met": ["TWF"], "severity": "alarm", "evidence": 11},
    "S05 오탐 경고": {"predicted": "HDF", "candidates": ["HDF"], "met": [], "severity": "warning", "evidence": 7},
}

# 관찰 포인트: 경로는 세 가지뿐이고, escalation 에는 반드시 ESC 코드가 사유로 붙습니다.
for practice_name, practice_card in practice_cards.items():
    print(f"  {practice_name:10} → {practice_route(practice_card)}")

**결과 해설** — S01 은 예측(HDF)과 판정이 같아 초안을 쓰고, S08 은 예측 OSF·판정 TWF 라 ESC-3 으로 넘기며, S05 는 판정 기준에 해당하는 것이 없는 경고라 오탐 점검(SOP-EA-01 8)으로 갑니다. 같은 카드는 언제나 같은 경로입니다.

### 2단계 · 경로마다 다른 노드, 실패하면 다시 쓰기

In [ ]:
# decide 뒤에서 경로마다 다른 노드로 갈라지고, 셋 다 validate 로 모입니다.
# validate 가 오류를 내면 draft 로 되돌아갑니다. 초안 대역은 미리 준 순서대로 초안을 냅니다.
# 관찰 포인트: trace 에 draft 와 validate 가 두 번씩 남습니다. 첫 초안은 없는 절(4.2.5)을 인용했습니다.
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph

PRACTICE_MAX_ATTEMPTS = 2

class PracticeState(TypedDict, total=False):
    card: dict
    route: str
    reasons: list
    draft: str
    attempts: int
    errors: list
    trace: Annotated[list, operator.add]

def practice_graph(drafts):
    queue = iter(drafts)
    def decide(state):
        route, reasons = practice_route(state["card"])
        return {"route": route, "reasons": reasons, "attempts": 0, "trace": [f"decide:{route}"]}
    def draft(state):
        attempts = state["attempts"] + 1
        return {"draft": next(queue), "attempts": attempts, "trace": [f"draft#{attempts}"]}
    def escalate(state):
        return {"draft": "", "trace": ["escalate"]}
    def inspect(state):
        return {"draft": "SOP-EA-01 8 에 따라 점검", "trace": ["inspect"]}
    def validate(state):
        errors = ["근거 목록에 없는 절: MC01-MM 4.2.5"] if "4.2.5" in state["draft"] else []
        return {"errors": errors, "trace": ["validate:" + ("오류" if errors else "통과")]}
    def after_validate(state):
        if state["errors"] and state["route"] == "grounded_draft" and state["attempts"] < PRACTICE_MAX_ATTEMPTS:
            return "draft"
        return "end"
    graph = StateGraph(PracticeState)
    for name, node in (("decide", decide), ("draft", draft), ("escalate", escalate),
                       ("inspect", inspect), ("validate", validate)):
        graph.add_node(name, node)
    graph.add_edge(START, "decide")
    graph.add_conditional_edges("decide", lambda s: s["route"], {
        "grounded_draft": "draft", "escalation": "escalate", "inspect_only": "inspect"})
    for name in ("draft", "escalate", "inspect"):
        graph.add_edge(name, "validate")
    graph.add_conditional_edges("validate", after_validate, {"draft": "draft", "end": END})
    return graph.compile()

practice_retried = practice_graph(["4.2.5 의 절차를 따른다", "4.2.3 의 절차를 따른다"]).invoke(
    {"card": practice_cards["S01 HDF 정탐"], "trace": []})
print("S01:", " → ".join(practice_retried["trace"]), "· 남은 오류", practice_retried["errors"])
for practice_name in ("S08 예측≠판정", "S05 오탐 경고"):
    practice_other = practice_graph([]).invoke({"card": practice_cards[practice_name], "trace": []})
    print(f"{practice_name[:3]}:", " → ".join(practice_other["trace"]))

**결과 해설** — S01 은 첫 초안이 없는 절을 인용해 validate 가 오류를 냈고, draft 로 되돌아가 두 번째 초안이 통과했습니다. S08·S05 는 초안 노드를 거치지 않고 escalate·inspect 에서 바로 validate 로 갑니다. 문장을 쓰는 것은 grounded_draft 경로뿐입니다.

### 3단계 · 계속 틀리면 상한에서 멈춘다

In [ ]:
# 초안 대역이 늘 같은 오류를 냅니다. 되돌아가는 길에 상한이 없으면 끝나지 않습니다.
# 상한에 걸리면 오류를 단 채 끝납니다. 오류를 지우고 통과시키지 않습니다.
# 관찰 포인트: attempts 가 상한(2)에서 멈추고, errors 가 그대로 남아 다음 단계(사람 검토)로 갑니다.
practice_bounded = practice_graph(["4.2.5 의 절차를 따른다"] * 10).invoke(
    {"card": practice_cards["S01 HDF 정탐"], "trace": []})
print("계속 틀린 초안:", " → ".join(practice_bounded["trace"]))
print("시도", practice_bounded["attempts"], "번 · 남은 오류", practice_bounded["errors"])

assert practice_route(practice_cards["S01 HDF 정탐"]) == ("grounded_draft", [])
assert practice_route(practice_cards["S08 예측≠판정"]) == ("escalation", ["ESC-3"])
assert practice_route(practice_cards["S05 오탐 경고"]) == ("inspect_only", ["SOP-EA-01 8"])
assert practice_retried["trace"] == ["decide:grounded_draft", "draft#1", "validate:오류", "draft#2", "validate:통과"]
assert practice_bounded["attempts"] == PRACTICE_MAX_ATTEMPTS and practice_bounded["errors"], "상한에서 멈추지 않았거나 오류가 사라졌습니다"

**결과 해설** — 늘 틀리는 초안도 두 번 쓰고 멈췄고, 오류는 그대로 남았습니다. 상한에 걸렸다고 오류를 지우면 검사하지 않은 것과 같습니다. 앱은 이렇게 오류가 남은 초안을 사람에게 보내되, **승인은 할 수 없게** 막습니다(Notebook 11).

## 직접 해 볼 과제
`practice_try_card` 의 값을 바꿔 경로가 어떻게 달라지는지 보세요. 후보를 `["HDF", "PWF"]` 로 늘리면? `severity` 를 "alarm" 으로 바꾸고 `met` 를 비우면? `evidence` 를 0 으로 만들면? 사유가 둘 이상 붙는 카드도 만들어 보세요.

In [ ]:
# 카드 하나를 바꿔 규칙에 넣습니다. 그래프 없이 규칙만 봅니다.
# 사유는 걸린 조건마다 하나씩 붙습니다.
practice_try_card = {"predicted": "HDF", "candidates": ["HDF"], "met": ["HDF"], "severity": "warning", "evidence": 9}
print(practice_route(practice_try_card))

## 중간 결과
카드 셋의 경로와 사유, 검사 실패 뒤 다시 써서 통과한 흐름, 경로마다 지나간 노드, 그리고 늘 틀리는 초안이 상한에서 오류를 단 채 멈추는 것을 확인합니다.

## 실패 경계
ESC-1(현장 안전)과 ESC-6(설비 정지 요구)은 카드만으로 판단할 수 없어 규칙에 없습니다. 이것은 현장에서 사람이 판단합니다. 또 규칙이 grounded_draft 로 보낸 카드가 실제 고장이라는 보장은 없습니다. TWF 는 위험 구간에서 무작위로 일어나 기준이 성립해도 오탐일 수 있습니다(Notebook 12 에서 숫자로 봅니다).

## 실제 app 연결
과제 10 App 의 `routing.py::decide_route` 가 같은 규칙(ESC-2·3·4·5·7)이고, ESC-5(조치 완료 뒤 24시간 안 재발)는 요청에 함께 온 조치 완료 기록으로 판단합니다. 되돌아가는 조건은 `review.py::after_validate`, 상한은 `review.py::MAX_ATTEMPTS`(2)입니다.

## 다음 Notebook 연결
다음 `10_parallel_collectors.ipynb` 에서는 매뉴얼 근거와 이력 근거를 두 담당이 동시에 모으고 합칩니다.